# SASV: ECAPA + CM score-sum (ASVspoof 2019 LA)

Baseline1-style fusion:

```text
s_sasv = s_asv + (1 - P_spoof)
```

- `s_asv` = ECAPA cosine (enrol model vs test)
- `P_spoof` = your app LA detector (**lfcc** or **wavlm**)

Run **`02_ecapa_only_sasv.ipynb` first** so you can compare EERs.

Use lab LA flacs (not browser mics). WavLM is fine here; the browser ~1.0 issue does not apply to ASVspoof files.

In [5]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "score_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR
from score_lib import score_fused_trials

print("cuda:", torch.cuda.is_available())

cuda: True


## Knobs

- `CM_BACKEND = "lfcc"` — app LFCC-LA checkpoint
- `CM_BACKEND = "wavlm"` — your WavLM+ASP checkpoint
- Keep `SMOKE = True` until the loop works

In [6]:
SMOKE = False
SPLIT = "dev"
MAX_TRIALS = 500 if SMOKE else 0
CM_BACKEND = "lfcc"   # or "wavlm"
DEVICE = "cuda"
FORCE_CPU = False

## Run fusion scoring

In [7]:
summary = score_fused_trials(
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    split=SPLIT,
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    cm_backend=CM_BACKEND,
    output_dir=RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_{SPLIT}",
)
{
    "system": summary["system"],
    "sasv_eer_%": summary["sasv_eer_percent"],
    "sv_eer_%": summary["sv_eer_percent"],
    "spf_eer_%": summary["spf_eer_percent"],
    "n": summary["num_scored"],
}

Trials: {'target': 1484, 'nontarget': 5768, 'spoof': 22296, 'total': 29548} | CM=lfcc


Could not parse CUDA device string 'cuda': not enough values to unpack (expected 2, got 1). Falling back to device 0.


Enrol dev:   0%|          | 0/10 [00:00<?, ?it/s]

Score fused:   0%|          | 0/29548 [00:00<?, ?it/s]

{
  "system": "ecapa_plus_lfcc_sum",
  "split": "dev",
  "max_trials": 0,
  "num_scored": 29548,
  "key_counts": {
    "target": 1484,
    "nontarget": 5768,
    "spoof": 22296,
    "total": 29548
  },
  "device": "cuda",
  "cm_backend": "lfcc",
  "fusion": "s_asv + (1 - p_spoof)",
  "sasv_eer": 0.01143814139022684,
  "sv_eer": 0.02097780859973302,
  "spf_eer": 0.0008970218880723312,
  "sasv_eer_percent": 1.143814139022684,
  "sv_eer_percent": 2.097780859973302,
  "spf_eer_percent": 0.08970218880723312
}


{'system': 'ecapa_plus_lfcc_sum',
 'sasv_eer_%': 1.143814139022684,
 'sv_eer_%': 2.097780859973302,
 'spf_eer_%': 0.08970218880723312,
 'n': 29548}

## Compare with ECAPA-only

If you already ran notebook 02, load both metrics JSONs:

In [8]:
import json

ecapa_path = RUNS_DIR / f"ecapa_only_{SPLIT}" / f"metrics_{SPLIT}.json"
fused_path = RUNS_DIR / f"ecapa_plus_{CM_BACKEND}_{SPLIT}" / f"metrics_{SPLIT}.json"

if ecapa_path.exists():
    ecapa = json.loads(ecapa_path.read_text(encoding="utf-8"))
    print("ECAPA-only SASV-EER %:", ecapa["sasv_eer_percent"])
    print("ECAPA-only SPF-EER %:", ecapa["spf_eer_percent"])
else:
    print("No ECAPA-only metrics at", ecapa_path)

fused = json.loads(fused_path.read_text(encoding="utf-8"))
print("Fused SASV-EER %:", fused["sasv_eer_percent"])
print("Fused SPF-EER %:", fused["spf_eer_percent"])

ECAPA-only SASV-EER %: 15.229110512138206
ECAPA-only SPF-EER %: 17.909041980507457
Fused SASV-EER %: 1.143814139022684
Fused SPF-EER %: 0.08970218880723312
